# YOLO26-Seg Multi-Crop Foliar Lesion Segmentation & Localization Pipeline (v1 — Overnight Master Notebook)

**Project**: AI Powered Crop Intelligence Rover  
**Scope**: Train an independent, human-annotation-grounded **YOLO26-Seg** multi-crop disease lesion instance segmentation & detection model (`nc=1: disease_lesion`) to upgrade internal disease spot localization beyond the legacy PlantVillage Tomato Early Blight `YOLOv8n` lesion detector (`yolov8n_lesions_best.pt`) and HSV-trained `Mobile-UNet` (`mobile_unet_best.pt`), without overwriting any existing production checkpoints or specimen models.

### Audited Dataset Integrity Policy
1. **Excluded Synthetic Pseudo-Labels**: `cv/datasets/processed/yolo_lesions` (`250 train / 80 val`) and `cv/datasets/processed/segmentation` (`200 train / 60 val`) were derived via HSV/LAB color thresholding on PlantVillage Tomato Early Blight and are **strictly excluded** from supervised training and evaluation.
2. **Primary Genuine Lesion Dataset — PlantSeg (`Wei et al., Scientific Data 2026`, Zenodo `17719108`, License `CC BY-NC 4.0`)**:
   - `7,774` in-the-wild plant disease images (`5,367` train / `846` val / `1,561` test) across **115 crop-disease categories** (`Corn Northern Leaf Blight`, `Corn Gray Leaf Spot`, `Corn Rust`, `Tomato Early/Late/Septoria/Bacterial/Mold/Viruses`, `Potato Early/Late Blight`, `Grape Black Rot/Downy Mildew`, `Apple Scab/Rust`, `Bell Pepper`, `Rice`, `Wheat`, `Soybean`, etc.).
   - Expert-reviewed human LabelMe polygons (`annotation_{train,val,test}.json`) + pixel masks (`annotations/{split}/*.png`).
3. **Field-Realistic Healthy Negative Controls**: Stratified healthy foliage from **PlantWild** (`CC BY-NC-ND 4.0`), **PlantDoc** (`MIT`), and **PlantVillage** (`CC0 1.0`) via your mounted `/kaggle/input` codebase dataset (`mohammadfayaskhan/smart-plant-codebase`), deduplicated across splits via MD5 + perceptual `dHash (Hamming <= 4)`.
4. **Non-Destructive Versioned Outputs**: All new artifacts write exclusively to `/kaggle/working/artifacts/yolo26_lesion_v1/` and `/kaggle/working/runs/yolo26_lesion_v1/`, finishing with a downloadable `/kaggle/working/yolo26_lesion_v1_release_bundle.zip`.

> **Kaggle Overnight Run Instructions**: Select **GPU T4** (or **T4 x2**) + **Internet ON**, attach `mohammadfayaskhan/smart-plant-codebase` (recommended for healthy negative controls & legacy baseline evaluation), and click **Run All** / **Save & Run All (Commit)** (~2.5–3.5 hours unattended with automatic `last.pt` resume).
        

In [ ]:
# ==============================================================================
# CELL 1 — KAGGLE ENVIRONMENT, GPU/VRAM PROBE & OFFICIAL YOLO26-SEG CHECK
# ==============================================================================
import os
import sys
import json
import time
import shutil
import platform
import importlib
import subprocess
from datetime import datetime, timezone
from pathlib import Path

os.environ["WANDB_DISABLED"] = "true"
os.environ["WANDB_MODE"] = "disabled"
os.environ["YOLO_VERBOSE"] = "False"
os.environ["TOKENIZERS_PARALLELISM"] = "false"

WORK_ROOT = Path("/kaggle/working") if Path("/kaggle/working").exists() else Path.cwd()
ARTIFACTS_DIR = WORK_ROOT / "artifacts" / "yolo26_lesion_v1"
RUNS_DIR = WORK_ROOT / "runs" / "yolo26_lesion_v1"
DATASET_STAGE_DIR = WORK_ROOT / "datasets" / "yolo26_lesion_v1"

for sub in ["checkpoints", "configs", "datasets", "evaluation", "pretrained"]:
    (ARTIFACTS_DIR / sub).mkdir(parents=True, exist_ok=True)
RUNS_DIR.mkdir(parents=True, exist_ok=True)
DATASET_STAGE_DIR.mkdir(parents=True, exist_ok=True)

def ensure_ultralytics():
    need_install = False
    try:
        import ultralytics
        parts = tuple(int(x) for x in ultralytics.__version__.split(".")[:2])
        if parts < (8, 4):
            need_install = True
    except Exception:
        need_install = True
    if need_install:
        print("Installing/upgrading ultralytics>=8.4.0 for official YOLO26-Seg support...")
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "--upgrade", "ultralytics>=8.4.0"])
        importlib.invalidate_caches()
    import ultralytics
    return ultralytics.__version__

ul_ver = ensure_ultralytics()

import numpy as np
import pandas as pd
import cv2
import torch
from PIL import Image
from ultralytics import YOLO

print("=" * 88)
print("YOLO26-SEG LESION PIPELINE v1 — CELL 1 ENVIRONMENT & ARCHITECTURE CHECK")
print("=" * 88)

cuda_ok = torch.cuda.is_available()
gpu_name = torch.cuda.get_device_name(0) if cuda_ok else "CPU"
num_gpus = torch.cuda.device_count() if cuda_ok else 0
vram_gb = round(torch.cuda.get_device_properties(0).total_memory / (1024 ** 3), 2) if cuda_ok else 0.0
cap = torch.cuda.get_device_capability(0) if cuda_ok else (0, 0)

def disk_free_gb(path: Path) -> float:
    return round(shutil.disk_usage(str(path)).free / (1024 ** 3), 2)

tmp_root = Path("/kaggle/tmp") if Path("/kaggle/tmp").exists() else Path("/tmp")
print(f"  Platform      : {platform.platform()}")
print(f"  PyTorch       : {torch.__version__} | CUDA={torch.version.cuda} | GPU={gpu_name} (sm_{cap[0]}{cap[1]}, {vram_gb} GB, count={num_gpus})")
print(f"  Ultralytics   : {ul_ver} | OpenCV={cv2.__version__} | NumPy={np.__version__}")
print(f"  Disk Free     : working={disk_free_gb(WORK_ROOT)} GB | tmp={disk_free_gb(tmp_root)} GB")

arch_checks = {}
YOLO26_MODE_SUPPORT = {"nms_flag": False}
dummy_img = np.zeros((256, 256, 3), dtype=np.uint8)

for cfg in ["yolo26n-seg.yaml", "yolo26s-seg.yaml"]:
    m = YOLO(cfg)
    inner = m.model
    head = inner.model[-1]
    rec = {
        "cfg": cfg,
        "head_class": type(head).__name__,
        "reg_max": getattr(head, "reg_max", None),
        "yaml_end2end": bool(getattr(inner, "yaml", {}).get("end2end", False)),
        "head_end2end_init": bool(getattr(head, "end2end", False)),
        "one2many": hasattr(head, "cv2") and hasattr(head, "cv3"),
        "one2one": hasattr(head, "one2one_cv2") and hasattr(head, "one2one_cv3"),
        "parameters": int(sum(p.numel() for p in inner.parameters())),
        "predict_nms_false_ok": False,
        "predict_nms_true_ok": False,
    }
    try:
        m.predict(dummy_img, imgsz=256, nms=False, verbose=False)
        rec["predict_nms_false_ok"] = True
        YOLO26_MODE_SUPPORT["nms_flag"] = True
    except Exception:
        pass
    try:
        m.predict(dummy_img, imgsz=256, nms=True, verbose=False)
        rec["predict_nms_true_ok"] = True
        YOLO26_MODE_SUPPORT["nms_flag"] = True
    except Exception:
        pass
    arch_checks[cfg] = rec
    print(
        f"  ✓ {cfg:<16} | head={rec['head_class']:<10} | reg_max={rec['reg_max']} | "
        f"o2o={rec['one2one']} | params={rec['parameters']:,} | nms_false={rec['predict_nms_false_ok']} | nms_true={rec['predict_nms_true_ok']}"
    )
    del m

if cuda_ok:
    torch.cuda.empty_cache()

env_prov = {
    "timestamp_utc": datetime.now(timezone.utc).strftime("%Y-%m-%dT%H:%M:%SZ"),
    "torch": torch.__version__,
    "cuda": torch.version.cuda,
    "ultralytics": ul_ver,
    "gpu_name": gpu_name,
    "compute_capability": f"sm_{cap[0]}{cap[1]}",
    "vram_gb": vram_gb,
    "num_gpus": num_gpus,
    "working_free_gb": disk_free_gb(WORK_ROOT),
    "tmp_free_gb": disk_free_gb(tmp_root),
    "arch_checks": arch_checks,
    "mode_support": YOLO26_MODE_SUPPORT,
}
(ARTIFACTS_DIR / "configs" / "env_provenance.json").write_text(json.dumps(env_prov, indent=2), encoding="utf-8")
print("✓ Cell 1 complete -> env_provenance.json saved.")
        

In [ ]:
# ==============================================================================
# CELL 2 — MOUNTED DATASET DISCOVERY & ZENODO PLANTSEG (17719108) ACQUISITION
# ==============================================================================
import hashlib
import zipfile
import urllib.request

PLANTSEG_ZENODO_URL = "https://zenodo.org/api/records/17719108/files/plantseg.zip/content"
PLANTSEG_EXPECTED_MD5 = "9358a66dff88cdd15c4fe009763c40a3"

print("=" * 88)
print("CELL 2 — DATASET DISCOVERY & PLANTSEG VERIFIED ACQUISITION")
print("=" * 88)

def scan_input_paths():
    found = {
        "codebase_root": None,
        "plantdoc_root": None,
        "plantvillage_root": None,
        "plantwild_root": None,
        "plantseg_root": None,
        "legacy_yolov8_lesions": None,
        "legacy_mobile_unet": None,
        "yolo26_specimen_best": None,
        "pretrained_yolo26n_lesion": None,
        "pretrained_yolo26s_lesion": None,
        "test_images_dir": None,
        "samples_dir": None,
    }
    for sr in [Path("/kaggle/input"), WORK_ROOT]:
        if not sr.exists():
            continue
        for root, dirs, files in os.walk(sr):
            rp = Path(root)
            if len(rp.parts) - len(sr.parts) > 8:
                dirs[:] = []
                continue
            dirs[:] = [d for d in dirs if not d.startswith(".") and d not in ("__pycache__", ".git", "node_modules")]
            fset = set(files)
            dset = set(dirs)
            name_l = rp.name.lower()

            if found["codebase_root"] is None and ("backend" in dset and "cv" in dset):
                found["codebase_root"] = rp
            if found["legacy_yolov8_lesions"] is None and "yolov8n_lesions_best.pt" in fset:
                found["legacy_yolov8_lesions"] = rp / "yolov8n_lesions_best.pt"
            if found["legacy_mobile_unet"] is None and "mobile_unet_best.pt" in fset:
                found["legacy_mobile_unet"] = rp / "mobile_unet_best.pt"
            if found["yolo26_specimen_best"] is None and "yolo26_specimen_best.pt" in fset:
                found["yolo26_specimen_best"] = rp / "yolo26_specimen_best.pt"
            if found["pretrained_yolo26n_lesion"] is None and "yolo26n_lesion_640_best.pt" in fset:
                found["pretrained_yolo26n_lesion"] = rp / "yolo26n_lesion_640_best.pt"
            if found["pretrained_yolo26s_lesion"] is None and "yolo26s_lesion_768_best.pt" in fset:
                found["pretrained_yolo26s_lesion"] = rp / "yolo26s_lesion_768_best.pt"
            if found["test_images_dir"] is None and name_l == "test_images" and any(f.lower().endswith((".jpg", ".png")) for f in files):
                found["test_images_dir"] = rp
            if found["samples_dir"] is None and name_l == "samples" and any(f.lower().endswith(".jpg") for f in files):
                found["samples_dir"] = rp

            if found["plantseg_root"] is None:
                if ("annotation_train.json" in fset or "Metadata.csv" in fset) and ("images" in dset and "annotations" in dset):
                    found["plantseg_root"] = rp
            if found["plantdoc_root"] is None:
                xml_cnt = sum(1 for f in files if f.lower().endswith(".xml"))
                if ("plantdoc" in str(rp).lower()) and (("train" in dset and "test" in dset) or xml_cnt >= 50):
                    found["plantdoc_root"] = rp
            if found["plantvillage_root"] is None and name_l in ("color", "plantvillage", "plantvillage_dataset"):
                if len([d for d in dirs if "___" in d]) >= 15:
                    found["plantvillage_root"] = rp
            if found["plantwild_root"] is None and "plantwild" in name_l:
                if len([d for d in dirs if any(k in d.lower() for k in ("leaf", "blight", "rust"))]) >= 15:
                    found["plantwild_root"] = rp
                elif "images" in dset and len([d for d in (rp / "images").iterdir() if d.is_dir()]) >= 15:
                    found["plantwild_root"] = rp / "images"
    return found

DISCOVERED = scan_input_paths()
for k, v in DISCOVERED.items():
    print(f"  {k:<22}: {v}")

SCRATCH_DIR = Path("/kaggle/tmp") if (Path("/kaggle/tmp").exists() and shutil.disk_usage("/kaggle/tmp").free > 6 * 1024**3) else (WORK_ROOT / "_tmp_plantseg")
SCRATCH_DIR.mkdir(parents=True, exist_ok=True)
PLANTSEG_LOCAL_ROOT = SCRATCH_DIR / "plantseg_extracted"

def find_plantseg_dir(base: Path):
    for root, dirs, files in os.walk(base):
        rp = Path(root)
        if ("images" in dirs and "annotations" in dirs) or ("annotation_train.json" in files):
            return rp
    return None

def md5_file(path: Path) -> str:
    h = hashlib.md5()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(1024 * 1024), b""):
            h.update(chunk)
    return h.hexdigest()

def download_with_resume(url: str, dst: Path, expected_md5: str):
    if dst.exists() and md5_file(dst) == expected_md5:
        print(f"  ✓ Cached plantseg.zip matches expected MD5 ({expected_md5}).")
        return
    existing = dst.stat().st_size if dst.exists() else 0
    headers = {"User-Agent": "YOLO26-Lesion-Trainer/1.0"}
    if existing > 0:
        headers["Range"] = f"bytes={existing}-"
    req = urllib.request.Request(url, headers=headers)
    with urllib.request.urlopen(req, timeout=60) as resp:
        code = getattr(resp, "status", 200)
        mode = "ab" if (code == 206 and existing > 0) else "wb"
        if mode == "wb":
            existing = 0
        total = int(resp.headers.get("Content-Length", 0)) + existing
        print(f"  Downloading PlantSeg from Zenodo (~{total/1024**2:.1f} MB) -> {dst} ...")
        downloaded = existing
        next_log = 128 * 1024 * 1024
        with open(dst, mode) as out:
            while True:
                buf = resp.read(4 * 1024 * 1024)
                if not buf:
                    break
                out.write(buf)
                downloaded += len(buf)
                if downloaded >= next_log or (total > 0 and downloaded >= total):
                    print(f"    ... {downloaded/1024**2:.1f}/{total/1024**2:.1f} MB ({downloaded/max(1,total)*100:.1f}%)")
                    next_log += 128 * 1024 * 1024
    got_md5 = md5_file(dst)
    assert got_md5 == expected_md5, f"PlantSeg zip MD5 mismatch: got {got_md5}, expected {expected_md5}"
    print(f"  ✓ Download verified: MD5={got_md5}")

if DISCOVERED["plantseg_root"] is None:
    cached = find_plantseg_dir(PLANTSEG_LOCAL_ROOT) if PLANTSEG_LOCAL_ROOT.exists() else None
    if cached is not None:
        DISCOVERED["plantseg_root"] = cached
        print(f"  ✓ Using already-extracted PlantSeg at {cached}")
    else:
        zip_path = SCRATCH_DIR / "plantseg.zip"
        download_with_resume(PLANTSEG_ZENODO_URL, zip_path, PLANTSEG_EXPECTED_MD5)
        PLANTSEG_LOCAL_ROOT.mkdir(parents=True, exist_ok=True)
        print(f"  Extracting {zip_path.name} -> {PLANTSEG_LOCAL_ROOT} ...")
        with zipfile.ZipFile(zip_path, "r") as zf:
            zf.extractall(PLANTSEG_LOCAL_ROOT)
        if zip_path.exists():
            zip_path.unlink()
        DISCOVERED["plantseg_root"] = find_plantseg_dir(PLANTSEG_LOCAL_ROOT)

assert DISCOVERED["plantseg_root"] is not None, "PlantSeg dataset could not be located or downloaded!"
print(f"✓ Final PlantSeg root: {DISCOVERED['plantseg_root']}")
        

In [ ]:
# ==============================================================================
# CELL 3 — GENUINE VS SYNTHETIC ANNOTATION AUDIT & PLANTSEG SCHEMA ANALYSIS
# ==============================================================================
from collections import Counter

PLANTSEG_ROOT = Path(DISCOVERED["plantseg_root"])
print("=" * 88)
print("CELL 3 — DATA INTEGRITY AUDIT & PLANTSEG LESION COVERAGE")
print("=" * 88)

excluded_synthetic = []
if DISCOVERED["codebase_root"] is not None:
    for rel in ["cv/datasets/processed/yolo_lesions", "cv/datasets/processed/segmentation"]:
        p = Path(DISCOVERED["codebase_root"]) / rel
        if p.exists():
            cnt = sum(1 for _ in p.rglob("*.jpg")) + sum(1 for _ in p.rglob("*.png"))
            excluded_synthetic.append({"path": str(p), "files": cnt, "reason": "Synthetic HSV/LAB thresholding on PlantVillage Tomato___Early_blight — EXCLUDED"})
            print(f"  [EXCLUDED] {rel} ({cnt} files) — synthetic HSV/LAB pseudo-labels")

audit_report = {
    "dataset": "PlantSeg (Wei et al., Scientific Data 2026, Zenodo 17719108)",
    "license": "CC BY-NC 4.0",
    "human_annotation_method": "Expert-reviewed LabelMe polygons + pixel segmentation masks",
    "excluded_synthetic_datasets": excluded_synthetic,
    "splits": {},
    "priority_crop_coverage": {},
}

meta_csv = PLANTSEG_ROOT / "Metadata.csv"
if meta_csv.exists():
    df_meta = pd.read_csv(meta_csv)
    print(f"  Metadata.csv rows: {len(df_meta)} | cols: {list(df_meta.columns)}")

priority_keywords = [
    "corn_northern_leaf_blight", "corn_gray_leaf_spot", "corn_rust",
    "tomato_early_blight", "tomato_late_blight", "tomato_septoria", "tomato_bacterial",
    "tomato_leaf_mold", "potato_early_blight", "potato_late_blight",
    "grape_black_rot", "grape_downy_mildew", "apple_scab", "apple_rust", "apple_black_rot",
    "bell_pepper", "soybean", "rice", "wheat",
]
crop_counts = Counter()
priority_counts = Counter()
total_imgs, total_masks, total_coco_anns, tiny_under_1pct, small_under_2pct = 0, 0, 0, 0, 0

for sp in ["train", "val", "test"]:
    img_dir = PLANTSEG_ROOT / "images" / sp
    mask_dir = PLANTSEG_ROOT / "annotations" / sp
    coco_p = PLANTSEG_ROOT / f"annotation_{sp}.json"
    imgs = sorted([p for p in img_dir.glob("*") if p.suffix.lower() in (".jpg", ".jpeg", ".png", ".webp")]) if img_dir.exists() else []
    masks = sorted([p for p in mask_dir.glob("*.png")]) if mask_dir.exists() else []
    n_coco_imgs, n_coco_anns, n_poly, n_rle = 0, 0, 0, 0
    if coco_p.exists():
        cd = json.loads(coco_p.read_text(encoding="utf-8"))
        n_coco_imgs = len(cd.get("images", []))
        n_coco_anns = len(cd.get("annotations", []))
        im_area_map = {im["id"]: max(1, im.get("width", 1) * im.get("height", 1)) for im in cd.get("images", [])}
        for ann in cd.get("annotations", []):
            seg = ann.get("segmentation")
            if isinstance(seg, list):
                n_poly += 1
            elif isinstance(seg, dict):
                n_rle += 1
            area_frac = float(ann.get("area", 0.0)) / float(im_area_map.get(ann.get("image_id"), 1))
            if area_frac < 0.01:
                tiny_under_1pct += 1
            if area_frac < 0.02:
                small_under_2pct += 1
    for p in imgs:
        crop = p.stem.split("_")[0].lower()
        crop_counts[crop] += 1
        sl = p.stem.lower()
        for pk in priority_keywords:
            if pk in sl:
                priority_counts[pk] += 1
    total_imgs += len(imgs)
    total_masks += len(masks)
    total_coco_anns += n_coco_anns
    audit_report["splits"][sp] = {
        "disk_images": len(imgs),
        "disk_png_masks": len(masks),
        "coco_json_exists": coco_p.exists(),
        "coco_images": n_coco_imgs,
        "coco_annotations": n_coco_anns,
        "polygon_anns": n_poly,
        "rle_anns": n_rle,
    }
    print(f"  PlantSeg [{sp:<5}] imgs={len(imgs):>5} | png_masks={len(masks):>5} | coco_imgs={n_coco_imgs:>5} | coco_anns={n_coco_anns:>5} (poly={n_poly}, rle={n_rle})")

audit_report["totals"] = {
    "images": total_imgs,
    "masks": total_masks,
    "coco_annotations": total_coco_anns,
    "anns_under_1pct_area": tiny_under_1pct,
    "anns_under_2pct_area": small_under_2pct,
    "unique_crop_prefixes": len(crop_counts),
}
audit_report["top_20_crops"] = crop_counts.most_common(20)
audit_report["priority_crop_coverage"] = dict(sorted(priority_counts.items()))

print(f"  Total PlantSeg: {total_imgs} images, {total_masks} PNG masks, {total_coco_anns} COCO annotations")
print(f"  Small lesions: {tiny_under_1pct} (<1% area), {small_under_2pct} (<2% area) -> imgsz=768 + erasing=0.0!")
print("  Priority pathology counts:", dict(priority_counts.most_common(12)))
(ARTIFACTS_DIR / "configs" / "cell3_lesion_dataset_audit.json").write_text(json.dumps(audit_report, indent=2), encoding="utf-8")
print("✓ Cell 3 audit saved.")
        

In [ ]:
# ==============================================================================
# CELL 4 — ZERO-LEAKAGE PLANTSEG CONVERSION + FIELD NEGATIVE CONTROLS (YOLO-SEG)
# ==============================================================================
import random
import xml.etree.ElementTree as ET
import matplotlib.pyplot as plt

IMG_EXTS = {".jpg", ".jpeg", ".png", ".webp", ".bmp"}
random.seed(42)
np.random.seed(42)

print("=" * 88)
print("CELL 4 — PLANTSEG COCO/MASK -> YOLO26-SEG CONVERSION & DEDUP")
print("=" * 88)

def compute_md5_and_dhash(img_path: Path):
    h_md5 = hashlib.md5(img_path.read_bytes()).hexdigest()
    try:
        with Image.open(img_path) as im:
            w, h = im.size
            if w < 16 or h < 16:
                return h_md5, None, None
            gray = im.convert("L").resize((9, 8), Image.Resampling.BILINEAR)
            arr = np.asarray(gray, dtype=np.int16)
            bits = (arr[:, 1:] > arr[:, :-1]).flatten()
            val = 0
            for b in bits:
                val = (val << 1) | int(bool(b))
            return h_md5, val, (w, h)
    except Exception:
        return h_md5, None, None

def mask_to_yolo_polygons(mask_arr: np.ndarray, class_id: int = 0, min_area_px: float = 12.0):
    h, w = mask_arr.shape[:2]
    if h <= 0 or w <= 0:
        return []
    bin_mask = (mask_arr > 0).astype(np.uint8)
    contours, _ = cv2.findContours(bin_mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    polys = []
    for cnt in contours:
        if len(cnt) < 3 or cv2.contourArea(cnt) < min_area_px:
            continue
        peri = cv2.arcLength(cnt, True)
        approx = cv2.approxPolyDP(cnt, max(0.65, 0.003 * peri), True)
        if len(approx) < 3:
            approx = cnt
        if len(approx) < 3:
            continue
        pts = []
        for x, y in approx.reshape(-1, 2):
            pts.append(max(0.0, min(1.0, float(x) / float(w))))
            pts.append(max(0.0, min(1.0, float(y) / float(h))))
        if len(pts) >= 6:
            polys.append((class_id, pts))
    return polys

def poly_area_px(flat_xy) -> float:
    if len(flat_xy) < 6:
        return 0.0
    pts = np.asarray(flat_xy, dtype=np.float32).reshape(-1, 2)
    return float(cv2.contourArea(pts))

def rle_to_mask(rle: dict, h: int, w: int) -> np.ndarray:
    try:
        from pycocotools import mask as coco_mask
        if isinstance(rle.get("counts"), list):
            rle = coco_mask.frPyObjects(rle, h, w)
        return coco_mask.decode(rle).astype(np.uint8)
    except Exception:
        counts = rle.get("counts")
        if isinstance(counts, list):
            flat = np.zeros(h * w, dtype=np.uint8)
            idx, val = 0, 0
            for c in counts:
                if val == 1:
                    flat[idx:idx + c] = 1
                idx += c
                val = 1 - val
            return flat.reshape((w, h)).T
        return np.zeros((h, w), dtype=np.uint8)

def parse_coco_and_mask_polygons(anns: list, img_w: int, img_h: int, mask_path: Path):
    polys = []
    used_mask_fallback = False
    for ann in anns:
        if int(ann.get("iscrowd", 0)) == 1:
            used_mask_fallback = True
            continue
        seg = ann.get("segmentation")
        if isinstance(seg, list):
            for sub in seg:
                if not isinstance(sub, list) or len(sub) < 6:
                    continue
                if poly_area_px(sub) < 12.0:
                    continue
                norm = []
                for k in range(0, len(sub) - 1, 2):
                    nx = max(0.0, min(1.0, float(sub[k]) / float(img_w)))
                    ny = max(0.0, min(1.0, float(sub[k + 1]) / float(img_h)))
                    norm.extend([nx, ny])
                if len(norm) >= 6:
                    polys.append((0, norm))
        elif isinstance(seg, dict):
            m = rle_to_mask(seg, img_h, img_w)
            polys.extend(mask_to_yolo_polygons(m, class_id=0, min_area_px=12.0))
    if not polys and mask_path is not None and mask_path.exists():
        m_arr = cv2.imread(str(mask_path), cv2.IMREAD_UNCHANGED)
        if m_arr is not None:
            if m_arr.ndim == 3:
                m_arr = m_arr[:, :, 0]
            polys = mask_to_yolo_polygons(m_arr, class_id=0, min_area_px=12.0)
            used_mask_fallback = True
    return polys, used_mask_fallback

def build_record_pool(split_name: str):
    img_dir = PLANTSEG_ROOT / "images" / split_name
    mask_dir = PLANTSEG_ROOT / "annotations" / split_name
    coco_p = PLANTSEG_ROOT / f"annotation_{split_name}.json"

    by_name, by_stem = {}, {}
    for p in sorted(img_dir.glob("*")):
        if p.suffix.lower() in IMG_EXTS:
            by_name[p.name] = p
            by_stem[p.stem] = p
    mask_by_stem = {p.stem: p for p in sorted(mask_dir.glob("*.png"))} if mask_dir.exists() else {}

    coco_anns_by_img_id = {}
    coco_images = []
    if coco_p.exists():
        cd = json.loads(coco_p.read_text(encoding="utf-8"))
        coco_images = cd.get("images", [])
        for a in cd.get("annotations", []):
            coco_anns_by_img_id.setdefault(a["image_id"], []).append(a)

    seen_disk_imgs = set()
    records = []
    for cim in coco_images:
        fn = Path(cim["file_name"]).name
        img_p = by_name.get(fn) or by_stem.get(Path(fn).stem)
        if img_p is None or img_p in seen_disk_imgs:
            continue
        seen_disk_imgs.add(img_p)
        md5_hex, dhash_int, dims = compute_md5_and_dhash(img_p)
        if dims is None:
            continue
        w, h = dims
        mp = mask_by_stem.get(img_p.stem)
        polys, _ = parse_coco_and_mask_polygons(coco_anns_by_img_id.get(cim["id"], []), w, h, mp)
        if not polys:
            continue
        records.append({
            "source": "PlantSeg",
            "img_path": img_p,
            "mask_path": mp,
            "w": w, "h": h,
            "polys": polys,
            "md5": md5_hex,
            "dhash": dhash_int,
            "disease_tag": "_".join(img_p.stem.split("_")[:-1]).lower() or img_p.stem.lower(),
            "crop_tag": img_p.stem.split("_")[0].lower(),
            "is_negative": False,
        })

    for stem, img_p in by_stem.items():
        if img_p in seen_disk_imgs:
            continue
        mp = mask_by_stem.get(stem)
        if mp is None:
            continue
        md5_hex, dhash_int, dims = compute_md5_and_dhash(img_p)
        if dims is None:
            continue
        m_arr = cv2.imread(str(mp), cv2.IMREAD_UNCHANGED)
        if m_arr is None:
            continue
        if m_arr.ndim == 3:
            m_arr = m_arr[:, :, 0]
        polys = mask_to_yolo_polygons(m_arr, class_id=0, min_area_px=12.0)
        if not polys:
            continue
        records.append({
            "source": "PlantSeg_PNG",
            "img_path": img_p,
            "mask_path": mp,
            "w": dims[0], "h": dims[1],
            "polys": polys,
            "md5": md5_hex,
            "dhash": dhash_int,
            "disease_tag": "_".join(img_p.stem.split("_")[:-1]).lower() or img_p.stem.lower(),
            "crop_tag": img_p.stem.split("_")[0].lower(),
            "is_negative": False,
        })
    return records

raw_test = build_record_pool("test")
raw_val = build_record_pool("val")
raw_train = build_record_pool("train")
print(f"  Raw valid PlantSeg images: train={len(raw_train)} | val={len(raw_val)} | test={len(raw_test)}")
assert len(raw_train) >= 1000 and len(raw_test) >= 300, "Unexpectedly low PlantSeg record count!"

def collect_healthy_negatives(max_pw=480, max_pd=180, max_pv=360):
    negs = []
    disease_kw = ("blight", "rust", "scab", "spot", "mildew", "rot", "mold", "virus", "smut", "canker", "curl", "wilt", "anthracnose", "mosaic", "scorch", "greening", "mite", "streak")
    if DISCOVERED["plantwild_root"] is not None:
        pw_candidates = []
        for cls_dir in sorted(Path(DISCOVERED["plantwild_root"]).iterdir()):
            if not cls_dir.is_dir():
                continue
            cname = cls_dir.name.lower()
            if "leaf" in cname and not any(dk in cname for dk in disease_kw):
                imgs = [p for p in sorted(cls_dir.glob("*")) if p.suffix.lower() in IMG_EXTS]
                random.shuffle(imgs)
                for p in imgs[:35]:
                    pw_candidates.append(("PlantWild", cname, p))
        random.shuffle(pw_candidates)
        for src, cname, p in pw_candidates[:max_pw]:
            md5_hex, dhash_int, dims = compute_md5_and_dhash(p)
            if dims is not None:
                negs.append({"source": src, "img_path": p, "mask_path": None, "w": dims[0], "h": dims[1], "polys": [], "md5": md5_hex, "dhash": dhash_int, "disease_tag": f"healthy_{cname}", "crop_tag": cname.split()[0], "is_negative": True})

    if DISCOVERED["plantdoc_root"] is not None:
        pd_root = Path(DISCOVERED["plantdoc_root"])
        pd_candidates = []
        for xml_p in sorted(pd_root.rglob("*.xml")):
            try:
                tree = ET.parse(xml_p)
                objs = [ob.findtext("name", "").strip().lower() for ob in tree.findall(".//object")]
                if objs and all(("leaf" in o and not any(dk in o for dk in disease_kw)) for o in objs):
                    img_p = next((xml_p.with_suffix(ext) for ext in [".jpg", ".jpeg", ".png", ".JPG", ".JPEG"] if xml_p.with_suffix(ext).exists()), None)
                    if img_p is not None:
                        split_hint = "test" if "test" in [x.lower() for x in xml_p.parts] else "train"
                        pd_candidates.append((split_hint, objs[0], img_p))
            except Exception:
                continue
        random.shuffle(pd_candidates)
        for sp_hint, cname, p in pd_candidates[:max_pd]:
            md5_hex, dhash_int, dims = compute_md5_and_dhash(p)
            if dims is not None:
                negs.append({"source": f"PlantDoc_{sp_hint}", "img_path": p, "mask_path": None, "w": dims[0], "h": dims[1], "polys": [], "md5": md5_hex, "dhash": dhash_int, "disease_tag": f"healthy_{cname}", "crop_tag": cname.split()[0], "is_negative": True, "split_hint": sp_hint})

    if DISCOVERED["plantvillage_root"] is not None:
        pv_candidates = []
        for cls_dir in sorted(Path(DISCOVERED["plantvillage_root"]).iterdir()):
            if cls_dir.is_dir() and "healthy" in cls_dir.name.lower():
                imgs = [p for p in sorted(cls_dir.glob("*")) if p.suffix.lower() in IMG_EXTS]
                random.shuffle(imgs)
                for p in imgs[:35]:
                    pv_candidates.append(("PlantVillage", cls_dir.name, p))
        random.shuffle(pv_candidates)
        for src, cname, p in pv_candidates[:max_pv]:
            md5_hex, dhash_int, dims = compute_md5_and_dhash(p)
            if dims is not None:
                negs.append({"source": src, "img_path": p, "mask_path": None, "w": dims[0], "h": dims[1], "polys": [], "md5": md5_hex, "dhash": dhash_int, "disease_tag": f"healthy_{cname.lower()}", "crop_tag": cname.split("_")[0].lower(), "is_negative": True})
    return negs

healthy_negs = collect_healthy_negatives()
print(f"  Healthy field/wild/studio negative candidates: {len(healthy_negs)}")

neg_train, neg_val, neg_test = [], [], []
for idx, r in enumerate(healthy_negs):
    hint = r.get("split_hint")
    if hint == "test":
        neg_test.append(r)
    elif hint == "train":
        neg_train.append(r)
    else:
        m = idx % 10
        if m < 7:
            neg_train.append(r)
        elif m == 7:
            neg_val.append(r)
        else:
            neg_test.append(r)

seen_md5 = set()
seen_dhash = []

def is_dup(rec, max_hamming=4) -> bool:
    if rec["md5"] in seen_md5:
        return True
    dh = rec["dhash"]
    if dh is not None:
        for prev in seen_dhash:
            if (dh ^ prev).bit_count() <= max_hamming:
                return True
    return False

def register_seen(rec):
    seen_md5.add(rec["md5"])
    if rec["dhash"] is not None:
        seen_dhash.append(rec["dhash"])

def dedup_split(pool, label_name):
    kept, dropped = [], 0
    for r in pool:
        if is_dup(r):
            dropped += 1
            continue
        register_seen(r)
        kept.append(r)
    print(f"  Dedup [{label_name:<10}]: kept={len(kept):>5} | dropped_leaks_or_dups={dropped:>4}")
    return kept

clean_test = dedup_split(raw_test + neg_test, "test")
clean_val = dedup_split(raw_val + neg_val, "val")
clean_train = dedup_split(raw_train + neg_train, "train")

for sp in ("train", "val", "test"):
    (DATASET_STAGE_DIR / "images" / sp).mkdir(parents=True, exist_ok=True)
    (DATASET_STAGE_DIR / "labels" / sp).mkdir(parents=True, exist_ok=True)

stage_summary = {"splits": {}, "instances": {}, "negative_controls": {}, "corrupt_reencoded": 0}
image_metadata_index = []

for sp, recs in [("test", clean_test), ("val", clean_val), ("train", clean_train)]:
    sp_imgs, sp_inst, sp_negs = 0, 0, 0
    for idx, r in enumerate(recs):
        safe_tag = "".join(ch if (ch.isalnum() or ch in "._-") else "_" for ch in r["img_path"].stem[:36])
        prefix = "neg" if r["is_negative"] else "ps"
        stem = f"{sp}_{prefix}_{idx:05d}_{safe_tag}"
        dst_img = DATASET_STAGE_DIR / "images" / sp / f"{stem}.jpg"
        dst_lbl = DATASET_STAGE_DIR / "labels" / sp / f"{stem}.txt"

        try:
            if r["img_path"].suffix.lower() in (".jpg", ".jpeg"):
                shutil.copy2(r["img_path"], dst_img)
            else:
                with Image.open(r["img_path"]) as im:
                    im.convert("RGB").save(dst_img, format="JPEG", quality=95)
                stage_summary["corrupt_reencoded"] += 1
        except Exception:
            with Image.open(r["img_path"]) as im:
                im.convert("RGB").save(dst_img, format="JPEG", quality=95)
            stage_summary["corrupt_reencoded"] += 1

        lbl_lines = [
            f"{cid} " + " ".join(f"{v:.6f}" for v in pts)
            for cid, pts in r["polys"]
        ]
        dst_lbl.write_text("\n".join(lbl_lines) + ("\n" if lbl_lines else ""), encoding="utf-8")

        sp_imgs += 1
        sp_inst += len(lbl_lines)
        if r["is_negative"]:
            sp_negs += 1

        if sp in ("val", "test"):
            image_metadata_index.append({
                "split": sp,
                "staged_image": str(dst_img),
                "staged_label": str(dst_lbl),
                "gt_png_mask": str(r["mask_path"]) if r["mask_path"] else None,
                "source": r["source"],
                "crop_tag": r["crop_tag"],
                "disease_tag": r["disease_tag"],
                "is_negative": r["is_negative"],
                "gt_instances": len(lbl_lines),
                "width": r["w"],
                "height": r["h"],
            })

    stage_summary["splits"][sp] = sp_imgs
    stage_summary["instances"][sp] = sp_inst
    stage_summary["negative_controls"][sp] = sp_negs

yaml_path = DATASET_STAGE_DIR / "yolo26_lesion_v1.yaml"
yaml_path.write_text(
    "\n".join([
        f"path: {DATASET_STAGE_DIR.resolve()}",
        "train: images/train",
        "val: images/val",
        "test: images/test",
        "names:",
        "  0: 'disease_lesion'",
    ]) + "\n",
    encoding="utf-8",
)
stage_summary["yaml_path"] = str(yaml_path)

(ARTIFACTS_DIR / "configs" / "cell4_prep_summary.json").write_text(json.dumps(stage_summary, indent=2), encoding="utf-8")
(ARTIFACTS_DIR / "configs" / "cell4_image_metadata.json").write_text(json.dumps(image_metadata_index), encoding="utf-8")
print("✓ Staged YOLO26-Seg dataset summary:", json.dumps(stage_summary, indent=2))

vis_candidates = [
    m for m in image_metadata_index
    if m["split"] == "test" and not m["is_negative"] and any(k in m["disease_tag"] for k in ("corn_northern", "tomato_early", "potato", "grape_black", "apple_scab"))
]
random.shuffle(vis_candidates)
vis_negs = [m for m in image_metadata_index if m["split"] == "test" and m["is_negative"]][:2]
vis_pick = vis_candidates[:6] + vis_negs

fig, axes = plt.subplots(2, 4, figsize=(18, 9))
for ax, rec in zip(axes.flatten(), vis_pick):
    im_bgr = cv2.imread(rec["staged_image"])
    im_rgb = cv2.cvtColor(im_bgr, cv2.COLOR_BGR2RGB)
    h, w = im_rgb.shape[:2]
    lbl_txt = Path(rec["staged_label"]).read_text(encoding="utf-8").strip().splitlines()
    overlay = im_rgb.copy()
    for line in lbl_txt:
        vals = [float(v) for v in line.split()[1:]]
        pts = np.asarray([[vals[i] * w, vals[i + 1] * h] for i in range(0, len(vals) - 1, 2)], dtype=np.int32)
        cv2.fillPoly(overlay, [pts], (255, 110, 40))
        cv2.polylines(im_rgb, [pts], True, (255, 60, 20), 2)
    blended = cv2.addWeighted(overlay, 0.28, im_rgb, 0.72, 0)
    ax.imshow(blended)
    ax.set_title(f"{rec['disease_tag'][:28]}\n({rec['gt_instances']} polygons)", fontsize=9)
    ax.axis("off")
for ax in axes.flatten()[len(vis_pick):]:
    ax.axis("off")
plt.tight_layout()
gt_vis_path = ARTIFACTS_DIR / "evaluation" / "cell4_gt_polygon_check.png"
plt.savefig(gt_vis_path, dpi=150)
plt.close(fig)
print(f"✓ GT polygon visual check saved -> {gt_vis_path}")

if SCRATCH_DIR.parent == WORK_ROOT and PLANTSEG_LOCAL_ROOT.exists():
    for d_drop in [PLANTSEG_ROOT / "images" / "train", PLANTSEG_ROOT / "annotations" / "train", PLANTSEG_ROOT / "images" / "val", PLANTSEG_ROOT / "images" / "test"]:
        if d_drop.exists():
            shutil.rmtree(d_drop, ignore_errors=True)
    print(f"✓ Cleaned transient raw images; free working disk = {disk_free_gb(WORK_ROOT)} GB")
        

In [ ]:
# ==============================================================================
# CELL 5 — OFFICIAL PRETRAINED YOLO26-SEG WEIGHTS & T4 VRAM BENCHMARK
# ==============================================================================
print("=" * 88)
print("CELL 5 — OFFICIAL PRETRAINED CHECK & DYNAMIC T4 BATCH/VRAM PROBE")
print("=" * 88)

PRETRAINED_DIR = ARTIFACTS_DIR / "pretrained"
DATASET_YAML = Path(stage_summary["yaml_path"])

for w_name in ["yolo26n-seg.pt", "yolo26s-seg.pt"]:
    target_w = PRETRAINED_DIR / w_name
    if not target_w.exists():
        tmp_m = YOLO(w_name)
        src_w = Path(tmp_m.ckpt_path) if getattr(tmp_m, "ckpt_path", None) and Path(tmp_m.ckpt_path).exists() else Path(w_name)
        if src_w.exists():
            shutil.copy2(src_w, target_w)
        del tmp_m
    assert target_w.exists(), f"Missing official pretrained weights: {target_w}"
    sha = hashlib.sha256(target_w.read_bytes()).hexdigest()[:12]
    print(f"  ✓ Pretrained {w_name:<14} | {target_w.stat().st_size/1024**2:.2f} MB | sha={sha}")

probe_configs = [
    {"name": "probe_n_640_b16", "ckpt": "yolo26n-seg.pt", "imgsz": 640, "batch": 16},
    {"name": "probe_s_768_b8",  "ckpt": "yolo26s-seg.pt", "imgsz": 768, "batch": 8},
    {"name": "probe_s_768_b6",  "ckpt": "yolo26s-seg.pt", "imgsz": 768, "batch": 6},
]

hw_bench = []
SAFE_N_BATCH = 16
SAFE_S_IMGSZ = 768
SAFE_S_BATCH = 8

if DISCOVERED.get("pretrained_yolo26n_lesion") is not None and DISCOVERED.get("pretrained_yolo26s_lesion") is not None:
    print("✓ Mounted Version 1 lesion checkpoints detected -> skipping 10% VRAM probe.")
    probe_configs = []

for pc in probe_configs:
    if pc["name"] == "probe_s_768_b6" and SAFE_S_BATCH == 8:
        continue
    if cuda_ok:
        torch.cuda.reset_peak_memory_stats()
        torch.cuda.empty_cache()
    t0 = time.time()
    rec = dict(pc, status="PENDING")
    try:
        m = YOLO(str(PRETRAINED_DIR / pc["ckpt"]))
        m.train(
            data=str(DATASET_YAML),
            epochs=1,
            fraction=0.10,
            imgsz=pc["imgsz"],
            batch=pc["batch"],
            device=0 if cuda_ok else "cpu",
            amp=True,
            workers=2,
            cache=False,
            plots=False,
            val=False,
            project=str(RUNS_DIR / "_hw_probe"),
            name=pc["name"],
            exist_ok=True,
            verbose=False,
        )
        elapsed = time.time() - t0
        peak_gb = round(torch.cuda.max_memory_allocated() / (1024 ** 3), 2) if cuda_ok else 0.0
        rec.update({"status": "OK", "elapsed_10pct_s": round(elapsed, 2), "peak_vram_gb": peak_gb})
        if pc["name"] == "probe_s_768_b8" and peak_gb < (vram_gb * 0.85 if vram_gb > 0 else 99):
            SAFE_S_BATCH = 8
        del m
    except Exception as ex:
        rec.update({"status": f"OOM_OR_ERR: {str(ex)[:140]}"})
        if pc["ckpt"] == "yolo26n-seg.pt":
            SAFE_N_BATCH = 8
    hw_bench.append(rec)
    print(f"  {pc['name']}: {rec}")
    if cuda_ok:
        torch.cuda.empty_cache()

shutil.rmtree(RUNS_DIR / "_hw_probe", ignore_errors=True)
(ARTIFACTS_DIR / "configs" / "cell5_hw_benchmark.json").write_text(
    json.dumps({"safe_n_batch": SAFE_N_BATCH, "safe_s_imgsz": SAFE_S_IMGSZ, "safe_s_batch": SAFE_S_BATCH, "probes": hw_bench}, indent=2),
    encoding="utf-8",
)
print(f"✓ Cell 5 complete | Selected T4 config -> YOLO26n@640 batch={SAFE_N_BATCH} | YOLO26s@{SAFE_S_IMGSZ} batch={SAFE_S_BATCH}")
        

In [ ]:
# ==============================================================================
# CELL 6 — TRAIN EXPERIMENT 1: YOLO26n-SEG @ 640 (WITH AUTO-RESUME & VAL)
# ==============================================================================
EXP1_NAME = "yolo26n_lesion_640"

print("=" * 88)
print(f"CELL 6 — EXPERIMENT 1: {EXP1_NAME} (65 epochs, imgsz=640, batch={SAFE_N_BATCH})")
print("=" * 88)

def can_resume_last_ckpt(last_ckpt: Path, target_epochs: int) -> str:
    if not last_ckpt.exists():
        return "fresh"
    try:
        ck = torch.load(str(last_ckpt), map_location="cpu", weights_only=False)
        ep = int(ck.get("epoch", -99))
        if ep == -1 or (ep + 1) >= target_epochs:
            return "already_done"
        if 0 <= ep < (target_epochs - 1):
            return "resume"
    except Exception:
        pass
    return "fresh"

def train_with_resume_and_oom_guard(exp_name, pretrained_path, imgsz, init_batch, epochs, hyperparams, mounted_best=None):
    proj_dir = RUNS_DIR / "lesion_seg"
    run_dir = proj_dir / exp_name
    done_flag = ARTIFACTS_DIR / "configs" / f"{exp_name}_DONE.flag"
    best_dst = ARTIFACTS_DIR / "checkpoints" / f"{exp_name}_best.pt"
    last_ckpt = run_dir / "weights" / "last.pt"
    best_ckpt = run_dir / "weights" / "best.pt"

    if mounted_best is not None and Path(mounted_best).exists():
        shutil.copy2(mounted_best, best_dst)
        done_flag.write_text(json.dumps({"finished_utc": datetime.now(timezone.utc).strftime("%Y-%m-%dT%H:%M:%SZ"), "reused_from": str(mounted_best)}), encoding="utf-8")
        print(f"  ✓ Reused mounted trained checkpoint for {exp_name} -> {best_dst}")
        return best_dst

    if done_flag.exists() and best_dst.exists():
        print(f"  ✓ {exp_name} already finished in previous session -> {best_dst}")
        return best_dst

    t_start = time.time()
    batch = init_batch
    while batch >= 4:
        try:
            mode = can_resume_last_ckpt(last_ckpt, epochs)
            if mode == "already_done" and best_ckpt.exists():
                print(f"  ✓ {exp_name} reached final epoch inside previous run -> {best_ckpt}")
                break
            elif mode == "resume":
                print(f"  ▶ Resuming {exp_name} cleanly from interrupted checkpoint: {last_ckpt}")
                m = YOLO(str(last_ckpt))
                m.train(resume=True)
            else:
                print(f"  ▶ Starting fresh {exp_name} from {pretrained_path.name} (imgsz={imgsz}, batch={batch})")
                m = YOLO(str(pretrained_path))
                m.train(
                    data=str(DATASET_YAML),
                    epochs=epochs,
                    imgsz=imgsz,
                    batch=batch,
                    device=0 if cuda_ok else "cpu",
                    amp=True,
                    workers=2,
                    cache=False,
                    cos_lr=True,
                    patience=15,
                    save_period=10,
                    project=str(proj_dir),
                    name=exp_name,
                    exist_ok=True,
                    **hyperparams,
                )
            del m
            break
        except RuntimeError as exc:
            if "out of memory" in str(exc).lower() and batch > 4:
                print(f"  ⚠ CUDA OOM at batch={batch}; halving batch to {batch // 2} and cleaning run dir...")
                batch = max(4, batch // 2)
                shutil.rmtree(run_dir, ignore_errors=True)
                if cuda_ok:
                    torch.cuda.empty_cache()
            else:
                raise

    elapsed_min = round((time.time() - t_start) / 60.0, 2)
    assert best_ckpt.exists(), f"Training finished without best.pt at {best_ckpt}"
    shutil.copy2(best_ckpt, best_dst)
    if last_ckpt.exists():
        last_ckpt.unlink()
    done_flag.write_text(json.dumps({"finished_utc": datetime.now(timezone.utc).strftime("%Y-%m-%dT%H:%M:%SZ"), "elapsed_min": elapsed_min}), encoding="utf-8")
    print(f"  ✓ {exp_name} complete in {elapsed_min} min -> {best_dst}")
    return best_dst

EXP1_HYP = {
    "optimizer": "SGD",
    "lr0": 0.01,
    "lrf": 0.01,
    "warmup_epochs": 3.0,
    "close_mosaic": 12,
    "mosaic": 0.65,
    "mixup": 0.0,
    "copy_paste": 0.0,
    "erasing": 0.0,
    "hsv_h": 0.015,
    "hsv_s": 0.45,
    "hsv_v": 0.35,
    "degrees": 15.0,
    "translate": 0.10,
    "scale": 0.40,
    "fliplr": 0.50,
    "flipud": 0.15,
    "overlap_mask": True,
    "mask_ratio": 4,
}

exp1_best = train_with_resume_and_oom_guard(
    EXP1_NAME, PRETRAINED_DIR / "yolo26n-seg.pt", imgsz=640, init_batch=SAFE_N_BATCH, epochs=65, hyperparams=EXP1_HYP,
    mounted_best=DISCOVERED.get("pretrained_yolo26n_lesion"),
)

def evaluate_val_modes(ckpt_path: Path, imgsz: int, tag: str):
    import gc
    gc.collect()
    if cuda_ok:
        torch.cuda.empty_cache()
    report = {
        "experiment": tag,
        "checkpoint": str(ckpt_path),
        "sha256": hashlib.sha256(ckpt_path.read_bytes()).hexdigest(),
        "size_mb": round(ckpt_path.stat().st_size / 1024**2, 3),
        "imgsz": imgsz,
    }
    m = YOLO(str(ckpt_path))
    for mode_key, nms_val in [("val_e2e_nms_false", False), ("val_nms_true", True)]:
        res = m.val(data=str(DATASET_YAML), split="val", imgsz=imgsz, batch=2, half=bool(cuda_ok), nms=nms_val, plots=False, verbose=False)
        rd = res.results_dict
        report[mode_key] = {
            "box_p": round(float(rd.get("metrics/precision(B)", 0.0)), 4),
            "box_r": round(float(rd.get("metrics/recall(B)", 0.0)), 4),
            "box_map50": round(float(rd.get("metrics/mAP50(B)", 0.0)), 4),
            "box_map50_95": round(float(rd.get("metrics/mAP50-95(B)", 0.0)), 4),
            "mask_p": round(float(rd.get("metrics/precision(M)", 0.0)), 4),
            "mask_r": round(float(rd.get("metrics/recall(M)", 0.0)), 4),
            "mask_map50": round(float(rd.get("metrics/mAP50(M)", 0.0)), 4),
            "mask_map50_95": round(float(rd.get("metrics/mAP50-95(M)", 0.0)), 4),
            "speed_ms": {k: round(float(v), 2) for k, v in res.speed.items()},
        }
        print(f"  [{tag} | {mode_key}] Box mAP50={report[mode_key]['box_map50']} | Mask mAP50={report[mode_key]['mask_map50']} mAP50-95={report[mode_key]['mask_map50_95']}")
    del m
    gc.collect()
    if cuda_ok:
        torch.cuda.empty_cache()
    return report

exp1_val_rep = evaluate_val_modes(exp1_best, imgsz=640, tag=EXP1_NAME)
(ARTIFACTS_DIR / "configs" / f"{EXP1_NAME}_val_report.json").write_text(json.dumps(exp1_val_rep, indent=2), encoding="utf-8")
        

In [ ]:
# ==============================================================================
# CELL 7 — TRAIN EXPERIMENT 2: HIGH-RES YOLO26s-SEG @ 768 (PRIMARY CANDIDATE)
# ==============================================================================
EXP2_NAME = "yolo26s_lesion_768"

print("=" * 88)
print(f"CELL 7 — EXPERIMENT 2: {EXP2_NAME} (75 epochs, imgsz={SAFE_S_IMGSZ}, batch={SAFE_S_BATCH})")
print("=" * 88)

EXP2_HYP = dict(EXP1_HYP)
EXP2_HYP.update({
    "mosaic": 0.60,
    "close_mosaic": 12,
    "hsv_s": 0.40,
    "hsv_v": 0.30,
    "erasing": 0.0,
    "mixup": 0.0,
    "copy_paste": 0.0,
    "mask_ratio": 2,
})

exp2_best = train_with_resume_and_oom_guard(
    EXP2_NAME, PRETRAINED_DIR / "yolo26s-seg.pt", imgsz=SAFE_S_IMGSZ, init_batch=SAFE_S_BATCH, epochs=75, hyperparams=EXP2_HYP,
    mounted_best=DISCOVERED.get("pretrained_yolo26s_lesion"),
)
exp2_val_rep = evaluate_val_modes(exp2_best, imgsz=SAFE_S_IMGSZ, tag=EXP2_NAME)
(ARTIFACTS_DIR / "configs" / f"{EXP2_NAME}_val_report.json").write_text(json.dumps(exp2_val_rep, indent=2), encoding="utf-8")

for exp_tag in [EXP1_NAME, EXP2_NAME]:
    rdir = RUNS_DIR / "lesion_seg" / exp_tag
    for fn in ["results.png", "BoxPR_curve.png", "BoxF1_curve.png", "MaskPR_curve.png", "MaskF1_curve.png"]:
        src = rdir / fn
        if src.exists():
            shutil.copy2(src, ARTIFACTS_DIR / "evaluation" / f"train_{exp_tag}__{fn}")
print("✓ Cell 7 training & validation complete.")
        

In [ ]:
# ==============================================================================
# CELL 8 — VAL CONF CALIBRATION + FULL HELD-OUT TEST EVAL (VS YOLOV8n & MOBILE-UNET)
# ==============================================================================
import torch.nn as nn
from torchvision import transforms

print("=" * 88)
print("CELL 8 — VALIDATION CONF CALIBRATION & INDEPENDENT MULTI-CROP TEST SUITE")
print("=" * 88)

meta_idx = json.loads((ARTIFACTS_DIR / "configs" / "cell4_image_metadata.json").read_text(encoding="utf-8"))
val_meta = [m for m in meta_idx if m["split"] == "val"]
test_meta = [m for m in meta_idx if m["split"] == "test"]
print(f"  Eval Splits: val={len(val_meta)} (neg={sum(m['is_negative'] for m in val_meta)}) | test={len(test_meta)} (neg={sum(m['is_negative'] for m in test_meta)})")

PRIORITY_DISEASE_KEYS = [
    "corn_northern_leaf_blight", "corn_gray_leaf_spot", "corn_rust",
    "tomato_early_blight", "tomato_late_blight", "tomato_septoria",
    "potato_early_blight", "potato_late_blight",
    "grape_black_rot", "grape_downy_mildew", "apple_scab", "apple_rust",
]

class DoubleConv(nn.Module):
    def __init__(self, in_ch: int, out_ch: int):
        super().__init__()
        self.block = nn.Sequential(
            nn.Conv2d(in_ch, out_ch, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_ch, out_ch, kernel_size=3, padding=1, bias=False),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True),
        )
    def forward(self, x):
        return self.block(x)

class MobileUNet(nn.Module):
    def __init__(self, in_ch: int = 3, num_classes: int = 3):
        super().__init__()
        self.inc = DoubleConv(in_ch, 16)
        self.down1 = nn.Sequential(nn.MaxPool2d(2), DoubleConv(16, 32))
        self.down2 = nn.Sequential(nn.MaxPool2d(2), DoubleConv(32, 64))
        self.down3 = nn.Sequential(nn.MaxPool2d(2), DoubleConv(64, 128))
        self.up1 = nn.Upsample(scale_factor=2, mode="bilinear", align_corners=True)
        self.conv_up1 = DoubleConv(128 + 64, 64)
        self.up2 = nn.Upsample(scale_factor=2, mode="bilinear", align_corners=True)
        self.conv_up2 = DoubleConv(64 + 32, 32)
        self.up3 = nn.Upsample(scale_factor=2, mode="bilinear", align_corners=True)
        self.conv_up3 = DoubleConv(32 + 16, 16)
        self.outc = nn.Conv2d(16, num_classes, kernel_size=1)
    def forward(self, x):
        x1 = self.inc(x); x2 = self.down1(x1); x3 = self.down2(x2); x4 = self.down3(x3)
        d1 = self.conv_up1(torch.cat([self.up1(x4), x3], dim=1))
        d2 = self.conv_up2(torch.cat([self.up2(d1), x2], dim=1))
        d3 = self.conv_up3(torch.cat([self.up3(d2), x1], dim=1))
        return self.outc(d3)

def load_gt_boxes_and_mask(rec):
    h, w = rec["height"], rec["width"]
    lbl_lines = Path(rec["staged_label"]).read_text(encoding="utf-8").strip().splitlines()
    gt_boxes = []
    poly_mask = np.zeros((h, w), dtype=np.uint8)
    for line in lbl_lines:
        vals = [float(v) for v in line.split()[1:]]
        pts = np.asarray([[vals[i] * w, vals[i + 1] * h] for i in range(0, len(vals) - 1, 2)], dtype=np.float32)
        if len(pts) >= 3:
            x1, y1 = pts[:, 0].min(), pts[:, 1].min()
            x2, y2 = pts[:, 0].max(), pts[:, 1].max()
            if x2 > x1 and y2 > y1:
                gt_boxes.append([float(x1), float(y1), float(x2), float(y2)])
            cv2.fillPoly(poly_mask, [pts.astype(np.int32)], 1)
    gt_png = rec.get("gt_png_mask")
    if gt_png and Path(gt_png).exists():
        m = cv2.imread(gt_png, cv2.IMREAD_UNCHANGED)
        if m is not None:
            if m.ndim == 3:
                m = m[:, :, 0]
            if m.shape[:2] != (h, w):
                m = cv2.resize(m, (w, h), interpolation=cv2.INTER_NEAREST)
            return gt_boxes, (m > 0).astype(np.uint8)
    return gt_boxes, poly_mask

def box_iou_matrix(a: np.ndarray, b: np.ndarray) -> np.ndarray:
    if len(a) == 0 or len(b) == 0:
        return np.zeros((len(a), len(b)), dtype=np.float32)
    ix1 = np.maximum(a[:, None, 0], b[None, :, 0])
    iy1 = np.maximum(a[:, None, 1], b[None, :, 1])
    ix2 = np.minimum(a[:, None, 2], b[None, :, 2])
    iy2 = np.minimum(a[:, None, 3], b[None, :, 3])
    inter = np.maximum(0.0, ix2 - ix1) * np.maximum(0.0, iy2 - iy1)
    aa = (a[:, 2] - a[:, 0]) * (a[:, 3] - a[:, 1])
    ba = (b[:, 2] - b[:, 0]) * (b[:, 3] - b[:, 1])
    return inter / np.maximum(1e-6, aa[:, None] + ba[None, :] - inter)

def subset_key(rec) -> str:
    dt = rec["disease_tag"]
    for pk in PRIORITY_DISEASE_KEYS:
        if pk in dt:
            return pk
    return rec["crop_tag"]

def eval_detector_records(model_obj, records, imgsz, conf_thresh, nms_flag, label_name):
    import gc
    gc.collect()
    if cuda_ok:
        torch.cuda.empty_cache()
    tp_total, fp_total, fn_total = 0, 0, 0
    small1_gt, small1_hit, small2_gt, small2_hit = 0, 0, 0, 0
    neg_imgs, neg_fp_imgs, neg_fp_boxes = 0, 0, 0
    dice_list, iou_list, latencies_ms = [], [], []
    per_crop_stats = {}

    kw = {"imgsz": imgsz, "conf": conf_thresh, "iou": 0.45, "max_det": 150, "half": bool(cuda_ok), "verbose": False}
    if nms_flag is not None:
        kw["nms"] = nms_flag

    t_batch = time.time()
    for idx_rec, rec in enumerate(records):
        r0 = model_obj.predict(rec["staged_image"], **kw)[0]
        if cuda_ok and (idx_rec + 1) % 400 == 0:
            torch.cuda.empty_cache()
        sp_dict = getattr(r0, "speed", {}) or {}
        if "inference" in sp_dict:
            latencies_ms.append(float(sp_dict.get("preprocess", 0.0) + sp_dict.get("inference", 0.0) + sp_dict.get("postprocess", 0.0)))
        gt_boxes, gt_mask = load_gt_boxes_and_mask(rec)
        h, w = rec["height"], rec["width"]
        img_area = float(max(1, h * w))

        pred_boxes = r0.boxes.xyxy.cpu().numpy() if (r0.boxes is not None and len(r0.boxes) > 0) else np.zeros((0, 4), dtype=np.float32)
        pred_mask = np.zeros((h, w), dtype=np.uint8)
        if getattr(r0, "masks", None) is not None and getattr(r0.masks, "xy", None):
            for poly_xy in r0.masks.xy:
                if len(poly_xy) >= 3:
                    cv2.fillPoly(pred_mask, [np.round(poly_xy).astype(np.int32)], 1)

        if rec["is_negative"]:
            neg_imgs += 1
            if len(pred_boxes) > 0:
                neg_fp_imgs += 1
                neg_fp_boxes += len(pred_boxes)
                fp_total += len(pred_boxes)
            continue

        gt_arr = np.asarray(gt_boxes, dtype=np.float32) if gt_boxes else np.zeros((0, 4), dtype=np.float32)
        iou_mat = box_iou_matrix(gt_arr, pred_boxes)
        matched_gt, matched_pred = set(), set()
        if iou_mat.size > 0:
            pairs = sorted([(iou_mat[gi, pi], gi, pi) for gi in range(len(gt_arr)) for pi in range(len(pred_boxes)) if iou_mat[gi, pi] >= 0.50], reverse=True)
            for _, gi, pi in pairs:
                if gi not in matched_gt and pi not in matched_pred:
                    matched_gt.add(gi)
                    matched_pred.add(pi)

        tp = len(matched_gt)
        fp = len(pred_boxes) - tp
        fn = len(gt_arr) - tp
        tp_total += tp; fp_total += fp; fn_total += fn

        for gi, gb in enumerate(gt_boxes):
            af = ((gb[2] - gb[0]) * (gb[3] - gb[1])) / img_area
            if af < 0.02:
                small2_gt += 1
                if gi in matched_gt: small2_hit += 1
            if af < 0.01:
                small1_gt += 1
                if gi in matched_gt: small1_hit += 1

        gt_b, pr_b = (gt_mask > 0), (pred_mask > 0)
        inter = int(np.sum(gt_b & pr_b))
        union = int(np.sum(gt_b | pr_b))
        denom = int(np.sum(gt_b)) + int(np.sum(pr_b))
        im_dice = (2.0 * inter / denom) if denom > 0 else 1.0
        im_iou = (inter / union) if union > 0 else 1.0
        if getattr(r0, "masks", None) is not None or int(np.sum(gt_b)) > 0:
            dice_list.append(im_dice)
            iou_list.append(im_iou)

        key = subset_key(rec)
        st = per_crop_stats.setdefault(key, {"images": 0, "tp": 0, "fp": 0, "fn": 0, "dices": [], "ious": []})
        st["images"] += 1; st["tp"] += tp; st["fp"] += fp; st["fn"] += fn
        st["dices"].append(im_dice); st["ious"].append(im_iou)

    if not latencies_ms:
        latencies_ms = [((time.time() - t_batch) * 1000.0) / max(1, len(records))]

    p = tp_total / max(1, tp_total + fp_total)
    r = tp_total / max(1, tp_total + fn_total)
    f1 = (2 * p * r / max(1e-9, p + r))
    per_subset = {}
    for k, v in per_crop_stats.items():
        if v["images"] >= 5:
            sp_p = v["tp"] / max(1, v["tp"] + v["fp"])
            sp_r = v["tp"] / max(1, v["tp"] + v["fn"])
            per_subset[k] = {
                "images": v["images"],
                "precision": round(sp_p, 4),
                "recall": round(sp_r, 4),
                "f1": round(2 * sp_p * sp_r / max(1e-9, sp_p + sp_r), 4),
                "mean_dice": round(float(np.mean(v["dices"])), 4),
                "mean_iou": round(float(np.mean(v["ious"])), 4),
            }

    return {
        "model": label_name,
        "conf": conf_thresh,
        "nms": nms_flag,
        "precision_at_conf": round(p, 4),
        "recall_at_conf": round(r, 4),
        "f1_at_conf": round(f1, 4),
        "small_under_2pct_gt": small2_gt,
        "small_under_2pct_recall": round(small2_hit / max(1, small2_gt), 4),
        "small_under_1pct_gt": small1_gt,
        "small_under_1pct_recall": round(small1_hit / max(1, small1_gt), 4),
        "healthy_neg_images": neg_imgs,
        "healthy_fp_images": neg_fp_imgs,
        "healthy_fp_rate": round(neg_fp_imgs / max(1, neg_imgs), 4),
        "healthy_fp_boxes": neg_fp_boxes,
        "mean_mask_dice": round(float(np.mean(dice_list)) if dice_list else 0.0, 4),
        "mean_mask_iou": round(float(np.mean(iou_list)) if iou_list else 0.0, 4),
        "latency_mean_ms": round(float(np.mean(latencies_ms)), 2),
        "latency_p95_ms": round(float(np.percentile(latencies_ms, 95)), 2),
        "fps": round(1000.0 / max(1e-6, float(np.mean(latencies_ms))), 1),
        "per_subset_breakdown": dict(sorted(per_subset.items())),
    }

m_s = YOLO(str(exp2_best))
val_sweep = []
for c in [0.12, 0.15, 0.20, 0.25, 0.30, 0.35]:
    r_c = eval_detector_records(m_s, val_meta, SAFE_S_IMGSZ, c, nms_flag=True, label_name=f"val_s_{c}")
    val_sweep.append({"conf": c, "p": r_c["precision_at_conf"], "r": r_c["recall_at_conf"], "f1": r_c["f1_at_conf"], "dice": r_c["mean_mask_dice"], "healthy_fp": r_c["healthy_fp_rate"]})
    print(f"  [Val Sweep conf={c:.2f}] P={r_c['precision_at_conf']:.4f} R={r_c['recall_at_conf']:.4f} F1={r_c['f1_at_conf']:.4f} Dice={r_c['mean_mask_dice']:.4f} HealthyFP={r_c['healthy_fp_rate']*100:.1f}%")

eligible = [x for x in val_sweep if x["healthy_fp"] <= 0.15] or val_sweep
OPT_CONF = max(eligible, key=lambda x: x["f1"])["conf"]
print(f"✓ Selected optimal calibrated confidence threshold on VAL: OPT_CONF={OPT_CONF}")
del m_s
import gc
gc.collect()
if cuda_ok:
    torch.cuda.empty_cache()

test_results = {"val_conf_sweep": val_sweep, "optimal_conf": OPT_CONF, "models": {}}

def full_test_eval(ckpt_path: Path, imgsz: int, nms_flag, key_name: str):
    import gc
    gc.collect()
    if cuda_ok:
        torch.cuda.empty_cache()
    m = YOLO(str(ckpt_path))
    uv_kw = {"data": str(DATASET_YAML), "split": "test", "imgsz": imgsz, "batch": 2, "half": bool(cuda_ok), "plots": False, "verbose": False}
    if nms_flag is not None:
        uv_kw["nms"] = nms_flag
    uv = m.val(**uv_kw)
    rd = uv.results_dict
    custom = eval_detector_records(m, test_meta, imgsz, OPT_CONF if "yolo26" in key_name else 0.14, nms_flag=nms_flag, label_name=key_name)
    custom.update({
        "test_box_map50": round(float(rd.get("metrics/mAP50(B)", 0.0)), 4),
        "test_box_map50_95": round(float(rd.get("metrics/mAP50-95(B)", 0.0)), 4),
        "test_mask_map50": round(float(rd.get("metrics/mAP50(M)", 0.0)), 4),
        "test_mask_map50_95": round(float(rd.get("metrics/mAP50-95(M)", 0.0)), 4),
    })
    print(
        f"  [{key_name:<24}] Box mAP50={custom['test_box_map50']:.4f} | Mask mAP50={custom['test_mask_map50']:.4f} "
        f"| F1={custom['f1_at_conf']:.4f} | Dice={custom['mean_mask_dice']:.4f} IoU={custom['mean_mask_iou']:.4f} "
        f"| Small2%R={custom['small_under_2pct_recall']:.4f} | HealthyFP={custom['healthy_fp_rate']*100:.1f}% | {custom['latency_mean_ms']}ms"
    )
    del m
    gc.collect()
    if cuda_ok:
        torch.cuda.empty_cache()
    return custom

test_results["models"]["yolo26n_640_e2e"] = full_test_eval(exp1_best, 640, False, "yolo26n_640_e2e")
test_results["models"]["yolo26n_640_nms"] = full_test_eval(exp1_best, 640, True,  "yolo26n_640_nms")
test_results["models"]["yolo26s_768_e2e"] = full_test_eval(exp2_best, SAFE_S_IMGSZ, False, "yolo26s_768_e2e")
test_results["models"]["yolo26s_768_nms"] = full_test_eval(exp2_best, SAFE_S_IMGSZ, True,  "yolo26s_768_nms")

if DISCOVERED["legacy_yolov8_lesions"] is not None:
    test_results["models"]["legacy_yolov8n_256"] = full_test_eval(DISCOVERED["legacy_yolov8_lesions"], 256, None, "legacy_yolov8n_256")
    test_results["models"]["legacy_yolov8n_640"] = full_test_eval(DISCOVERED["legacy_yolov8_lesions"], 640, None, "legacy_yolov8n_640")

if DISCOVERED["legacy_mobile_unet"] is not None:
    unet = MobileUNet().to("cuda" if cuda_ok else "cpu").eval()
    sd = torch.load(str(DISCOVERED["legacy_mobile_unet"]), map_location="cuda" if cuda_ok else "cpu", weights_only=False)
    unet.load_state_dict(sd.get("model_state", sd))
    tf_unet = transforms.Compose([
        transforms.Resize((256, 256)),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
    ])
    unet_dices, unet_ious, unet_lats, unet_neg_fp, unet_neg_cnt = [], [], [], 0, 0
    for rec in test_meta:
        _, gt_mask = load_gt_boxes_and_mask(rec)
        h, w = rec["height"], rec["width"]
        pil_im = Image.open(rec["staged_image"]).convert("RGB")
        x_t = tf_unet(pil_im).unsqueeze(0).to("cuda" if cuda_ok else "cpu")
        t0 = time.time()
        with torch.inference_mode():
            pm256 = torch.argmax(unet(x_t), dim=1)[0].cpu().numpy().astype(np.uint8)
        unet_lats.append((time.time() - t0) * 1000.0)
        pm_les = cv2.resize((pm256 == 2).astype(np.uint8), (w, h), interpolation=cv2.INTER_NEAREST)
        if rec["is_negative"]:
            unet_neg_cnt += 1
            if int(np.sum(pm_les)) > 16:
                unet_neg_fp += 1
            continue
        gt_b, pr_b = (gt_mask > 0), (pm_les > 0)
        inter = int(np.sum(gt_b & pr_b))
        union = int(np.sum(gt_b | pr_b))
        denom = int(np.sum(gt_b)) + int(np.sum(pr_b))
        unet_dices.append((2.0 * inter / denom) if denom > 0 else 1.0)
        unet_ious.append((inter / union) if union > 0 else 1.0)
    test_results["models"]["legacy_mobile_unet_256"] = {
        "model": "legacy_mobile_unet_256",
        "mean_mask_dice": round(float(np.mean(unet_dices)), 4),
        "mean_mask_iou": round(float(np.mean(unet_ious)), 4),
        "healthy_neg_images": unet_neg_cnt,
        "healthy_fp_images": unet_neg_fp,
        "healthy_fp_rate": round(unet_neg_fp / max(1, unet_neg_cnt), 4),
        "latency_mean_ms": round(float(np.mean(unet_lats)), 2),
        "fps": round(1000.0 / max(1e-6, float(np.mean(unet_lats))), 1),
    }
    print(f"  [legacy_mobile_unet_256  ] Dice={test_results['models']['legacy_mobile_unet_256']['mean_mask_dice']:.4f} IoU={test_results['models']['legacy_mobile_unet_256']['mean_mask_iou']:.4f} | HealthyFP={test_results['models']['legacy_mobile_unet_256']['healthy_fp_rate']*100:.1f}%")
    del unet

y26_candidates = [k for k in test_results["models"] if k.startswith("yolo26")]
winner_key = max(y26_candidates, key=lambda k: (test_results["models"][k]["test_mask_map50"], test_results["models"][k]["f1_at_conf"], test_results["models"][k]["mean_mask_dice"]))
winner_ckpt = exp2_best if "yolo26s" in winner_key else exp1_best
WINNER_PATH = ARTIFACTS_DIR / "checkpoints" / "yolo26_lesion_best.pt"
shutil.copy2(winner_ckpt, WINNER_PATH)
WINNER_IMGSZ = SAFE_S_IMGSZ if "yolo26s" in winner_key else 640
WINNER_NMS = winner_key.endswith("_nms")

test_results["selected_winner"] = {
    "winner_variant": winner_key,
    "checkpoint": str(WINNER_PATH),
    "sha256": hashlib.sha256(WINNER_PATH.read_bytes()).hexdigest(),
    "recommended_imgsz": WINNER_IMGSZ,
    "recommended_nms_flag": WINNER_NMS,
    "recommended_conf_thresh": OPT_CONF,
}
(ARTIFACTS_DIR / "configs" / "cell8_lesion_test_report.json").write_text(json.dumps(test_results, indent=2), encoding="utf-8")
print(f"✓ WINNER SELECTED: {winner_key} -> {WINNER_PATH} (imgsz={WINNER_IMGSZ}, nms={WINNER_NMS}, conf={OPT_CONF})")
        

In [ ]:
# ==============================================================================
# CELL 9 — MULTI-CROP & FIELD VISUAL COMPARISON (GT vs YOLOv8n vs U-NET vs YOLO26-SEG)
# ==============================================================================
print("=" * 88)
print("CELL 9 — VISUAL COMPARISON GRIDS (MULTICROP, FIELD & NEGATIVE CONTROLS)")
print("=" * 88)

m_win = YOLO(str(WINNER_PATH))
m_v8 = YOLO(str(DISCOVERED["legacy_yolov8_lesions"])) if DISCOVERED["legacy_yolov8_lesions"] is not None else None
if DISCOVERED["legacy_mobile_unet"] is not None:
    m_unet = MobileUNet().to("cuda" if cuda_ok else "cpu").eval()
    sd = torch.load(str(DISCOVERED["legacy_mobile_unet"]), map_location="cuda" if cuda_ok else "cpu", weights_only=False)
    m_unet.load_state_dict(sd.get("model_state", sd))
    tf_unet_vis = transforms.Compose([
        transforms.Resize((256, 256)),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
    ])
else:
    m_unet = None

def draw_yolo_pred_panel(img_bgr, model_obj, imgsz, conf, nms_flag=None, color=(255, 130, 30)):
    vis = img_bgr.copy()
    if model_obj is None:
        return vis, 0
    kw = {"imgsz": imgsz, "conf": conf, "iou": 0.45, "max_det": 100, "verbose": False}
    if nms_flag is not None:
        kw["nms"] = nms_flag
    r0 = model_obj.predict(img_bgr, **kw)[0]
    overlay = vis.copy()
    if getattr(r0, "masks", None) is not None and getattr(r0.masks, "xy", None):
        for poly_xy in r0.masks.xy:
            if len(poly_xy) >= 3:
                pts = np.round(poly_xy).astype(np.int32)
                cv2.fillPoly(overlay, [pts], color)
                cv2.polylines(vis, [pts], True, color, 2)
        vis = cv2.addWeighted(overlay, 0.30, vis, 0.70, 0)
    cnt = 0
    if r0.boxes is not None and len(r0.boxes) > 0:
        cnt = len(r0.boxes)
        for bx, cf in zip(r0.boxes.xyxy.cpu().numpy(), r0.boxes.conf.cpu().numpy()):
            x1, y1, x2, y2 = [int(round(v)) for v in bx]
            cv2.rectangle(vis, (x1, y1), (x2, y2), color, 2)
            cv2.putText(vis, f"{cf:.2f}", (x1, max(14, y1 - 4)), cv2.FONT_HERSHEY_SIMPLEX, 0.45, color, 1, cv2.LINE_AA)
    return vis, cnt

def draw_unet_panel(img_bgr):
    vis = img_bgr.copy()
    if m_unet is None:
        return vis, 0.0
    h, w = img_bgr.shape[:2]
    pil_im = Image.fromarray(cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB))
    x_t = tf_unet_vis(pil_im).unsqueeze(0).to("cuda" if cuda_ok else "cpu")
    with torch.inference_mode():
        pm = torch.argmax(m_unet(x_t), dim=1)[0].cpu().numpy().astype(np.uint8)
    pm_full = cv2.resize(pm, (w, h), interpolation=cv2.INTER_NEAREST)
    ov = vis.copy()
    ov[pm_full == 2] = (40, 60, 235)
    vis = cv2.addWeighted(ov, 0.42, vis, 0.58, 0)
    leaf_px = max(1, int(np.sum(pm_full >= 1)))
    dmg = round(100.0 * float(np.sum(pm_full == 2)) / leaf_px, 1)
    return vis, dmg

target_tags = ["corn_northern_leaf_blight", "tomato_early_blight", "potato_late_blight", "grape_black_rot", "apple_scab"]
picked_multicrop = []
for tt in target_tags:
    m_hit = next((r for r in test_meta if not r["is_negative"] and tt in r["disease_tag"] and r["gt_instances"] >= 2), None)
    if m_hit is not None:
        picked_multicrop.append(m_hit)

if picked_multicrop:
    fig, axes = plt.subplots(len(picked_multicrop), 4, figsize=(18, 4 * len(picked_multicrop)))
    if len(picked_multicrop) == 1:
        axes = np.expand_dims(axes, 0)
    for row_i, rec in enumerate(picked_multicrop):
        im_bgr = cv2.imread(rec["staged_image"])
        _, gt_m = load_gt_boxes_and_mask(rec)
        gt_vis = im_bgr.copy()
        ov_gt = gt_vis.copy(); ov_gt[gt_m > 0] = (0, 220, 80)
        gt_vis = cv2.addWeighted(ov_gt, 0.40, gt_vis, 0.60, 0)
        v8_vis, v8_c = draw_yolo_pred_panel(im_bgr, m_v8, 256, 0.14, None, color=(0, 140, 255))
        un_vis, dmg = draw_unet_panel(im_bgr)
        y26_vis, y26_c = draw_yolo_pred_panel(im_bgr, m_win, WINNER_IMGSZ, OPT_CONF, WINNER_NMS, color=(50, 200, 255))
        titles = [
            f"GT Mask: {rec['disease_tag']} ({rec['gt_instances']} inst)",
            f"Legacy YOLOv8n @256: {v8_c} boxes",
            f"Legacy Mobile-UNet @256: {dmg}% dmg",
            f"YOLO26-Seg ({winner_key}): {y26_c} inst",
        ]
        for col_j, (panel, ttl) in enumerate(zip([gt_vis, v8_vis, un_vis, y26_vis], titles)):
            axes[row_i, col_j].imshow(cv2.cvtColor(panel, cv2.COLOR_BGR2RGB))
            axes[row_i, col_j].set_title(ttl, fontsize=9)
            axes[row_i, col_j].axis("off")
    plt.tight_layout()
    plt.savefig(ARTIFACTS_DIR / "evaluation" / "cell9_plantseg_multicrop_comparison.png", dpi=150)
    plt.close(fig)
    print("✓ Saved cell9_plantseg_multicrop_comparison.png")

field_imgs = []
for d_key in ["test_images_dir", "samples_dir"]:
    if DISCOVERED[d_key] is not None:
        field_imgs.extend(sorted([p for p in Path(DISCOVERED[d_key]).glob("*") if p.suffix.lower() in (".jpg", ".jpeg", ".png")])[:6])

if field_imgs:
    field_imgs = field_imgs[:6]
    fig, axes = plt.subplots(len(field_imgs), 3, figsize=(14, 4 * len(field_imgs)))
    if len(field_imgs) == 1:
        axes = np.expand_dims(axes, 0)
    for r_i, fp in enumerate(field_imgs):
        im_bgr = cv2.imread(str(fp))
        v8_vis, v8_c = draw_yolo_pred_panel(im_bgr, m_v8, 256, 0.14, None, color=(0, 140, 255))
        un_vis, dmg = draw_unet_panel(im_bgr)
        y26_vis, y26_c = draw_yolo_pred_panel(im_bgr, m_win, WINNER_IMGSZ, OPT_CONF, WINNER_NMS, color=(50, 200, 255))
        for c_j, (pn, tt) in enumerate([(v8_vis, f"{fp.name} | YOLOv8n: {v8_c}"), (un_vis, f"Mobile-UNet: {dmg}%"), (y26_vis, f"YOLO26-Seg: {y26_c}")]):
            axes[r_i, c_j].imshow(cv2.cvtColor(pn, cv2.COLOR_BGR2RGB))
            axes[r_i, c_j].set_title(tt, fontsize=9)
            axes[r_i, c_j].axis("off")
    plt.tight_layout()
    plt.savefig(ARTIFACTS_DIR / "evaluation" / "cell9_field_test_images_comparison.png", dpi=150)
    plt.close(fig)
    print("✓ Saved cell9_field_test_images_comparison.png")

neg_vis_samples = [r for r in test_meta if r["is_negative"]][:6]
if neg_vis_samples:
    fig, axes = plt.subplots(len(neg_vis_samples), 2, figsize=(10, 3.6 * len(neg_vis_samples)))
    if len(neg_vis_samples) == 1:
        axes = np.expand_dims(axes, 0)
    for r_i, rec in enumerate(neg_vis_samples):
        im_bgr = cv2.imread(rec["staged_image"])
        v8_vis, v8_c = draw_yolo_pred_panel(im_bgr, m_v8, 256, 0.14, None, color=(0, 140, 255))
        y26_vis, y26_c = draw_yolo_pred_panel(im_bgr, m_win, WINNER_IMGSZ, OPT_CONF, WINNER_NMS, color=(50, 200, 255))
        axes[r_i, 0].imshow(cv2.cvtColor(v8_vis, cv2.COLOR_BGR2RGB)); axes[r_i, 0].set_title(f"{rec['disease_tag'][:24]} | YOLOv8n FP={v8_c}", fontsize=9); axes[r_i, 0].axis("off")
        axes[r_i, 1].imshow(cv2.cvtColor(y26_vis, cv2.COLOR_BGR2RGB)); axes[r_i, 1].set_title(f"YOLO26-Seg FP={y26_c}", fontsize=9); axes[r_i, 1].axis("off")
    plt.tight_layout()
    plt.savefig(ARTIFACTS_DIR / "evaluation" / "cell9_healthy_negative_controls_comparison.png", dpi=150)
    plt.close(fig)
    print("✓ Saved cell9_healthy_negative_controls_comparison.png")
        

In [ ]:
# ==============================================================================
# CELL 10 — TWO-STAGE PIPELINE CHECK + RELEASE MANIFEST & DOWNLOADABLE BUNDLE
# ==============================================================================
import zipfile

print("=" * 88)
print("CELL 10 — TWO-STAGE SPECIMEN+LESION SMOKE TEST & RELEASE BUNDLE")
print("=" * 88)

picked_mc = globals().get("picked_multicrop") or [r for r in test_meta if not r["is_negative"]][:3]
f_imgs = globals().get("field_imgs") or []

two_stage_checks = []
if DISCOVERED.get("yolo26_specimen_best") is not None:
    m_spec = YOLO(str(DISCOVERED["yolo26_specimen_best"]))
    probe_samples = [r["staged_image"] for r in picked_mc[:3]] + [str(p) for p in f_imgs[:3]]
    for sp_img in probe_samples:
        r_spec = m_spec.predict(sp_img, imgsz=768, conf=0.25, iou=0.45, agnostic_nms=True, nms=True, verbose=False)[0]
        r_les = m_win.predict(sp_img, imgsz=WINNER_IMGSZ, conf=OPT_CONF, iou=0.45, nms=WINNER_NMS, verbose=False)[0]
        s_boxes = r_spec.boxes.xyxy.cpu().numpy().tolist() if r_spec.boxes is not None else []
        l_boxes = r_les.boxes.xyxy.cpu().numpy().tolist() if r_les.boxes is not None else []
        contained = 0
        for lx1, ly1, lx2, ly2 in l_boxes:
            cx, cy = 0.5 * (lx1 + lx2), 0.5 * (ly1 + ly2)
            if not s_boxes or any((sx1 - 0.04*(sx2-sx1)) <= cx <= (sx2 + 0.04*(sx2-sx1)) and (sy1 - 0.04*(sy2-sy1)) <= cy <= (sy2 + 0.04*(sy2-sy1)) for sx1, sy1, sx2, sy2 in s_boxes):
                contained += 1
        two_stage_checks.append({"image": Path(sp_img).name, "specimen_boxes": len(s_boxes), "raw_lesion_boxes": len(l_boxes), "canopy_contained_lesions": contained})
        print(f"  [Two-Stage] {Path(sp_img).name:<34}: specimens={len(s_boxes)} | raw_lesions={len(l_boxes)} | contained={contained}")
    del m_spec

win_m = YOLO(str(WINNER_PATH))
win_head = win_m.model.model[-1]
manifest = {
    "manifest_version": "yolo26_lesion_v1",
    "created_utc": datetime.now(timezone.utc).strftime("%Y-%m-%dT%H:%M:%SZ"),
    "task": "Multi-Crop Foliar Disease Lesion Instance Segmentation & Localization",
    "training_dataset": {
        "positive_source": "PlantSeg (Wei et al., Scientific Data 2026, Zenodo 17719108)",
        "license": "CC BY-NC 4.0",
        "negative_controls": "PlantWild (CC BY-NC-ND 4.0) + PlantDoc (MIT) + PlantVillage (CC0 1.0)",
        "excluded_synthetic": [e["path"] for e in excluded_synthetic],
        "prep_summary": stage_summary,
    },
    "preserved_baselines_untouched": {
        "specimen_detector": "artifacts/yolo26_v1/checkpoints/yolo26_specimen_best.pt",
        "legacy_lesion_detector": "cv/models/yolov8n_lesions_best.pt",
        "legacy_segmenter": "cv/models/mobile_unet_best.pt",
        "classifier": "cv/models/efficientnetv2_s_best.pt",
    },
    "architecture": {
        "winner_variant": winner_key,
        "head_class": type(win_head).__name__,
        "reg_max": getattr(win_head, "reg_max", None),
        "has_one2one": hasattr(win_head, "one2one_cv2"),
        "nc": 1,
        "names": {0: "disease_lesion"},
        "recommended_imgsz": WINNER_IMGSZ,
        "recommended_nms_flag": WINNER_NMS,
        "recommended_conf_thresh": OPT_CONF,
    },
    "checkpoints": {},
    "val_reports": {
        EXP1_NAME: exp1_val_rep,
        EXP2_NAME: exp2_val_rep,
    },
    "test_evaluation": test_results,
    "two_stage_checks": two_stage_checks,
}
del win_m

for ck in sorted((ARTIFACTS_DIR / "checkpoints").glob("*.pt")):
    manifest["checkpoints"][ck.stem] = {
        "filename": ck.name,
        "relative_path": f"checkpoints/{ck.name}",
        "size_mb": round(ck.stat().st_size / 1024**2, 3),
        "sha256": hashlib.sha256(ck.read_bytes()).hexdigest(),
    }

manifest_path = ARTIFACTS_DIR / "RELEASE_MANIFEST_YOLO26_LESION_v1.json"
manifest_path.write_text(json.dumps(manifest, indent=2), encoding="utf-8")

bundle_zip = WORK_ROOT / "yolo26_lesion_v1_release_bundle.zip"
bundle_files = []
with zipfile.ZipFile(bundle_zip, "w", compression=zipfile.ZIP_DEFLATED) as zf:
    zf.write(manifest_path, arcname="RELEASE_MANIFEST_YOLO26_LESION_v1.json")
    bundle_files.append("RELEASE_MANIFEST_YOLO26_LESION_v1.json")
    for sub in ["checkpoints", "configs", "evaluation"]:
        for fp in sorted((ARTIFACTS_DIR / sub).rglob("*")):
            if fp.is_file() and not fp.name.endswith("_DONE.flag") and fp.name != "cell4_image_metadata.json":
                arc = str(fp.relative_to(ARTIFACTS_DIR))
                zf.write(fp, arcname=arc)
                bundle_files.append(arc)

print("=" * 88)
print("CELL 10 — YOLO26 LESION v1 RELEASE BUNDLE READY")
print("=" * 88)
print(f"  Manifest      : {manifest_path}")
print(f"  Bundle Archive: {bundle_zip} ({bundle_zip.stat().st_size/1024**2:.2f} MB, {len(bundle_files)} files)")
for ck_k, ck_v in manifest["checkpoints"].items():
    print(f"  • {ck_k:<24} | {ck_v['size_mb']:>6.2f} MB | sha256={ck_v['sha256'][:16]}...")
print("Download yolo26_lesion_v1_release_bundle.zip from Kaggle Output (/kaggle/working) when finished!")
        